# Notebook 1 — Environment verification & data download

Verifies Spark/HDFS, downloads raw datasets, uploads to HDFS under `hdfs_paths.BASE`, prints schema previews.


In [ ]:
import os
import sys
import subprocess

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("MusicTrend_01_Environment")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)
sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"Default parallelism: {sc.defaultParallelism}")
print(f"HDFS music root: {hp.BASE}")


In [ ]:
# HDFS connectivity check
r = subprocess.run(["hdfs", "dfs", "-ls", f"/user/{hp.USER}"], capture_output=True, text=True)
print(r.stdout or r.stderr)


In [ ]:
# Create HDFS directories
for d in [
    f"{hp.BASE}/raw/lastfm",
    f"{hp.BASE}/raw/spotify_charts",
    f"{hp.BASE}/raw/billboard",
    f"{hp.BASE}/raw/msd_audio",
    f"{hp.BASE}/processed/events",
    f"{hp.BASE}/processed/spotify",
    f"{hp.BASE}/processed/billboard",
    f"{hp.BASE}/processed/audio",
    f"{hp.BASE}/processed/features",
    f"{hp.BASE}/streaming/input",
    f"{hp.BASE}/streaming/output",
    f"{hp.BASE}/streaming/checkpoint",
    f"{hp.BASE}/models/rf_model",
]:
    subprocess.run(["hdfs", "dfs", "-mkdir", "-p", d], check=False)
print("Directories ensured.")


## Download datasets

Use `wget`/`curl` where URLs allow. **Kaggle** requires API token or browser download — if automated fetch fails, download `charts.csv` and `Hot 100.csv` locally and `hdfs dfs -put` into the paths below. Document any manual step in your run log.


In [ ]:
import urllib.request
import zipfile
import shutil

LOCAL = os.path.abspath("_downloads")
os.makedirs(LOCAL, exist_ok=True)

def fetch(url, dest):
    try:
        print("Fetching", url)
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception as e:
        print("Fetch failed:", e)
        return False

# Last.fm HetRec 2011
lf_zip = os.path.join(LOCAL, "hetrec2011-lastfm-2k.zip")
if fetch("https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip", lf_zip):
    with zipfile.ZipFile(lf_zip, "r") as z:
        z.extract("hetrec2011-lastfm-2k/user_artists.dat", LOCAL)
    subprocess.run(
        ["hdfs", "dfs", "-put", "-f", os.path.join(LOCAL, "hetrec2011-lastfm-2k/user_artists.dat"), hp.RAW_LASTFM],
        check=False,
    )

# Spotify / Billboard: often need Kaggle — placeholders; upload manually if needed
for label, path in [
    ("Spotify charts.csv", hp.RAW_SPOTIFY),
    ("Billboard Hot 100.csv", hp.RAW_BILLBOARD),
    ("MSD msd_audio_features.csv", hp.RAW_MSD),
]:
    print(f"If missing on HDFS, upload to: {path} ({label})")


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType, DateType,
)

# Explicit schemas — adjust if your CSV headers differ slightly

lastfm_schema = StructType(
    [
        StructField("userID", StringType(), True),
        StructField("artistID", StringType(), True),
        StructField("weight", StringType(), True),
    ]
)

spotify_schema = StructType(
    [
        StructField("title", StringType(), True),
        StructField("rank", IntegerType(), True),
        StructField("date", StringType(), True),
        StructField("artist", StringType(), True),
        StructField("url", StringType(), True),
        StructField("region", StringType(), True),
        StructField("chart", StringType(), True),
        StructField("trend", StringType(), True),
        StructField("streams", LongType(), True),
    ]
)

# Billboard export variants exist — keep string types for flexible parse
billboard_schema = StructType(
    [
        StructField("WeekID", StringType(), True),
        StructField("Song", StringType(), True),
        StructField("Performer", StringType(), True),
        StructField("SongID", StringType(), True),
        StructField("Instance", StringType(), True),
        StructField("Previous Week Position", StringType(), True),
        StructField("Peak Position", StringType(), True),
        StructField("Weeks on Chart", StringType(), True),
    ]
)

msd_schema = StructType(
    [
        StructField("artist_name", StringType(), True),
        StructField("song_title", StringType(), True),
        StructField("tempo", DoubleType(), True),
        StructField("energy", DoubleType(), True),
        StructField("loudness", DoubleType(), True),
        StructField("danceability", DoubleType(), True),
        StructField("key", StringType(), True),
        StructField("mode", StringType(), True),
    ]
)

def preview_csv(path, schema, desc, header=True, sep=","):
    try:
        df = spark.read.schema(schema).option("header", header).option("sep", sep).csv(path)
        print("===", desc, "===")
        df.printSchema()
        df.show(5, truncate=False)
        print("rows:", df.count())
    except Exception as e:
        print(desc, "not available yet:", e)

preview_csv(hp.RAW_LASTFM, lastfm_schema, "Last.fm", header=False, sep="\t")
preview_csv(hp.RAW_SPOTIFY, spotify_schema, "Spotify")
preview_csv(hp.RAW_BILLBOARD, billboard_schema, "Billboard")
preview_csv(hp.RAW_MSD, msd_schema, "MSD audio")


## Outputs confirmed

- Spark session and HDFS reachable.
- Raw paths defined in `hdfs_paths.py` populated (or documented for manual `hdfs dfs -put`).
- Schema previews executed for each dataset.
